In [2]:
puts `ls -l`

total 840
-rw-rw-r-- 1 osboxes osboxes 263521 Jan  7 13:20 biovista-diseases-2025.ipynb
-rw-rw-r-- 1 osboxes osboxes  11350 Jan  7 13:20 biovista-diseases-2026-Mondo.ipynb
-rw-rw-r-- 1 osboxes osboxes  59291 Jan 22 16:03 biovista-drug-2025.ipynb
-rw-rw-r-- 1 osboxes osboxes 206070 Jan  7 13:20 biovista-gene-2025.ipynb
-rw-rw-r-- 1 osboxes osboxes 145025 Jan  7 13:20 biovista-pathways-2025.ipynb
-rw-rw-r-- 1 osboxes osboxes  31822 Jan  7 13:20 biovista-phenotypes.ipynb
-rw-rw-r-- 1 osboxes osboxes  18912 Jan  7 13:20 BV Disease-Gene Graphing.ipynb
-rw-rw-r-- 1 osboxes osboxes  57522 Jan 22 18:06 BV Drug-Disease Graphing.ipynb
-rw-rw-r-- 1 osboxes osboxes  21002 Jan 22 18:03 BV Drug-Gene Graphing.ipynb
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 deprecated
-rw-rw-r-- 1 osboxes osboxes   1282 Jan  7 13:20 drug mapping strategy
drwxrwxr-x 3 osboxes osboxes   4096 Jan 22 18:06 graph
drwxrwxr-x 3 osboxes osboxes   4096 Jan 22 17:20 maps
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 m

In [1]:
puts `head -2 ./raw_data/bv-kg-20250225.large`

source_1	id_1	type_1	name_1	source_2	id_2	type_2	name_2	score	url
UMLS:Disease or Syndrome:MSH	C0268631	Disease	SSADH Deficiency	HP:human_phenotype	HP:0001263	Human Phenotype	Global developmental delay	0.0501869	https://www.biovista.com/db/link/%5B%5B%22Disease%7CSSADH%20Deficiency%22%5D,%20%5B%22Human%20Phenotype%7CGlobal%20developmental%20delay%22%5D%5D?strength-weight-map=%257B%2522MEDLINE_STRENGTH_AB%2522:1.0,%2522HPO%2522:100.0%257D


In [2]:
puts `cat ./maps/2026-biovista-disease-mondo.map`

biovista_umls,orphanet,snomed,name,mondo,mondo
C0027126,http://www.orpha.net/ORDO/Orphanet_273,http://purl.bioontology.org/ontology/SNOMEDCT/77956009,myotonic dystrophy,http://purl.obolibrary.org/obo/MONDO_0016107
C0349653,http://www.orpha.net/ORDO/Orphanet_79318,http://purl.bioontology.org/ontology/SNOMEDCT/459063003,PMM2-congenital disorder of glycosylation,http://purl.obolibrary.org/obo/MONDO_0008907
C0268467,http://www.orpha.net/ORDO/Orphanet_2102,http://purl.bioontology.org/ontology/SNOMEDCT/23447005,GTP cyclohydrolase I deficiency,http://purl.obolibrary.org/obo/MONDO_0100184
C0268631,http://www.orpha.net/ORDO/Orphanet_22,http://purl.bioontology.org/ontology/SNOMEDCT/49748000,succinic semialdehyde dehydrogenase deficiency,http://purl.obolibrary.org/obo/MONDO_0010083
C0043459,http://www.orpha.net/ORDO/Orphanet_912,http://purl.bioontology.org/ontology/SNOMEDCT/88469006,Zellweger spectrum disorders,http://purl.obolibrary.org/obo/MONDO_0019609
C0751882,http://www.orpha.net/ORDO/Orphan

In [3]:
require 'net/http'
require 'uri'
require 'json'

$hpo_cache = {}

def get_hpo_label(hp_code)
  return $hpo_cache[hp_code] if $hpo_cache.key?(hp_code)
  local   = hp_code.tr(':', '_').then { |s| s.start_with?('HP_') ? s : "HP_#{s}" }
  iri     = "http://purl.obolibrary.org/obo/#{local}"
  encoded = URI.encode_uri_component(URI.encode_uri_component(iri))
  uri     = URI("https://www.ebi.ac.uk/ols4/api/ontologies/hp/terms/#{encoded}")
  response = Net::HTTP.get_response(uri)
  result = if response.is_a?(Net::HTTPSuccess)
    data = JSON.parse(response.body)
    data['label'] || "no HPO match found for #{local}"
  else
    "no HPO match found for #{local}"
  end
  $hpo_cache[hp_code] = result
rescue => e
  $hpo_cache[hp_code] = "no HPO match found for #{local}"
end

:get_hpo_label

In [5]:
require 'linkeddata'
require 'csv'

graphing_errors = File.open('./graph/2026_phenotype-disease-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
disease_mappings = CSV.read('./maps/2026-biovista-disease-mondo.map', headers: true)
#  HPO terms are used as-is, no mapping requried.


# Create RDF graph
graph = RDF::Repository.new

failures = {}


# Process each entity relation
CSV.foreach('./raw_data/bv-kg-20250225.large', col_sep: "\t", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
        # Disease
        # Pathway
        # Drug
        # Human Phenotype
        # Gene
    #they have pheno-pheno relations!  LOL!  Surprise!
  next unless (row['type_1'] == "Disease" && row['type_2'] == "Human Phenotype") || (row['type_1'] == "Human Phenotype" && row['type_2'] == "Disease")
  if row['type_1'] == "Human Phenotype"
    pheno_id = row['id_1']
    pheno_label = row['name_1']
    disease_id = row['id_2']  
  else
    pheno_id = row['id_2']
    pheno_label = row['name_2']
    disease_id = row['id_1']
  end

      # warn "pheno_id #{pheno_id} disease_id #{disease_id}"
      # abort
      
  score = row['score']
  evidence = row['url']

  # HACK  - need to replace C0023264 with C2931891 to harmomnize with MONDO, it is 0023264 in the large datafile
  disease_id = 'C2931891' if disease_id == 'C0023264'
      
  # Find corresponding mappings
  if pheno_id.match(%r{(\d+)}) # HP:0001263 -> http://purl.obolibrary.org/obo/HP_0007221
      hpo = "http://purl.obolibrary.org/obo/HP_#{pheno_id.match(%r{(\d+)})[1]}"
      pheno_id = "HP_#{pheno_id.match(%r{(\d+)})[1]}"  # need to change the : to make the graph URI
  else
      warn "NO MATCH FOR PHENO ID #{pheno_id}"
      graphing_errors.write "Pheno lookup failed #{pheno_id}\n"
      next
  end
  disease_line = disease_mappings.find { |d| d['biovista_umls'] == disease_id }  # returns first row that matches
      
  unless disease_line
    next if failures[disease_id]
    failures[disease_id] = 1
    warn "disease lookup failed #{disease_id}"
    graphing_errors.write "disease lookup failed #{disease_id}\n"
    next
  end

      warn "disease_line #{disease_line.inspect}"

# biovista_umls,orphanet,snomed,name,mondo
# C0027126,http://www.orpha.net/ORDO/Orphanet_273,http://purl.bioontology.org/ontology/SNOMEDCT/77956009,MYOTONIC DYSTROPHY TYPE 1,http://purl.bioontology.org/ontology/MONDO/MONDO_0016107  snomed_uri = RDF::URI.new(disease['snomed'])
  mondo_uri = RDF::URI.new(disease_line["mondo"])
  mondo_type = RDF::URI.new("https://bioportal.bioontology.org/ontologies/MONDO") 
  mondo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Disease")
  mondo_label =  RDF::Literal.new("MONDO Term")
  #orphanet = RDF::URI.new(disease['orphanet'])
  disease_label = RDF::Literal.new(disease_line['name'])
  original_disease = RDF::Literal.new(disease_line['biovista_umls'])

  hpo = RDF::URI.new(hpo)
  hpo_type = RDF::URI.new("http://edamontology.org/data_3275")
  hpo_label =  RDF::Literal.new("HPO Phenotype Identifier")
  hpo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Phenotype")
  hpo_label = RDF::Literal.new(get_hpo_label(pheno_id))
  
  # Create context URI
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{pheno_id}_#{disease_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  

  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], hpo, graph_name: context_uri)
  graph << RDF::Statement.new(hpo, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)

  graph << RDF::Statement.new(mondo_uri, RDFS.label, disease_label, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_type, RDFS.label, mondo_label, graph_name: context_uri)

      
  graph << RDF::Statement.new(hpo,  RDFS.label,       hpo_label , graph_name: context_uri)
  graph << RDF::Statement.new(hpo,  RDF.type,         hpo_type, graph_name: context_uri)
  graph << RDF::Statement.new(hpo,  RDF.type,         hpo_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(hpo_type, RDFS.label,       RDF::Literal.new("HPO Ontology Term"), graph_name: context_uri)
  graph << RDF::Statement.new(hpo_core_type, RDFS.label,  RDF::Literal.new("Phenotype"), graph_name: context_uri)
  graph << RDF::Statement.new(hpo,  SIMPATHIC['original-id'], RDF::Literal.new("#{pheno_id}"), graph_name: context_uri)
      
  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Biovista"), graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'], RDF::URI.new(evidence), graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['score'], RDF::Literal.new(score), graph_name: general_context)
  graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'], RDF::Literal.new("ASSOCIATED_WITH"), graph_name: general_context)
      
end

# Write RDF to file in N-Quads format
File.open('./graph/2026_disease-phenotype.nq.large', 'w') do |f|
  RDF::Writer.for(:nquads).new(f) do |writer|
    writer << graph
  end
end
graphing_errors.close

puts "RDF quads written"

(irb):6: warning: already initialized constant Object::SIMPATHIC
(irb):6: warning: previous definition of SIMPATHIC was here
(irb):7: warning: already initialized constant Object::RDFS
(irb):7: warning: previous definition of RDFS was here
disease_line #<CSV::Row "biovista_umls":"C0268631" "orphanet":"http://www.orpha.net/ORDO/Orphanet_22" "snomed":"http://purl.bioontology.org/ontology/SNOMEDCT/49748000" "name":"succinic semialdehyde dehydrogenase deficiency" "mondo":"http://purl.obolibrary.org/obo/MONDO_0010083">
disease_line #<CSV::Row "biovista_umls":"C0268631" "orphanet":"http://www.orpha.net/ORDO/Orphanet_22" "snomed":"http://purl.bioontology.org/ontology/SNOMEDCT/49748000" "name":"succinic semialdehyde dehydrogenase deficiency" "mondo":"http://purl.obolibrary.org/obo/MONDO_0010083">
disease_line #<CSV::Row "biovista_umls":"C0268631" "orphanet":"http://www.orpha.net/ORDO/Orphanet_22" "snomed":"http://purl.bioontology.org/ontology/SNOMEDCT/49748000" "name":"succinic semialdehyde de

RDF quads written
